# EA Sports Player Performance Index


## Import libraries

In [1]:
# Ensure that no .pyc files are generated
import sys

sys.dont_write_bytecode = True

In [2]:
import warnings

import pandas as pd
from kloppy import statsbomb
from kloppy.domain import EventDataset
from config import paths, tournaments

In [3]:
warnings.filterwarnings(
    "ignore",
    message="The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    message="Boolean Series key will be reindexed to match DataFrame index.",
    category=UserWarning,
)

## Load providers data

In [4]:
# StatsBomb match ID for UEFA Euro 2024 Final
match_ids = tournaments.get_all_match_ids(tournaments.EURO_2024)
EURO_2024_FINAL_MATCH_ID = match_ids[0]

In [5]:
# Load dataset based on provider
dataset = statsbomb.load(
    event_data=paths.STATSBOMB_EVENTS_DIR / f"{EURO_2024_FINAL_MATCH_ID}.json",
    lineup_data=paths.STATSBOMB_LINEUPS_DIR / f"{EURO_2024_FINAL_MATCH_ID}.json",
)

## Convert dataset to DataFrame

In [6]:
# Convert dataset to DataFrame for easier exploration
df = dataset.to_df()  # type: ignore

In [7]:
# Display the first 5 rows of the dataset
df.head()

,event_id,event_type,period_id,timestamp,end_timestamp,ball_state,ball_owning_team,team_id,player_id,coordinates_x,...,end_coordinates_y,receiver_player_id,set_piece_type,body_part_type,pass_type,is_under_pressure,duel_type,is_counter_attack,goalkeeper_type,card_type
0,50aa204f-5d65-4145-8597-5d5628fb7898,GENERIC:Starting XI,1,0 days 00:00:00,NaT,alive,772,772,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
1,a279cbee-9ab3-4cfb-9c51-27cacc1bf2a2,GENERIC:Starting XI,1,0 days 00:00:00,NaT,alive,772,768,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
2,d2126e70-9f04-4bb7-ba2b-9377836d1757,GENERIC:Half Start,1,0 days 00:00:00,NaT,alive,772,768,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
3,54d78bfa-4146-42bd-acdc-97bcd393dd81,GENERIC:Half Start,1,0 days 00:00:00,NaT,alive,772,772,None,NaN,...,NaN,None,None,None,None,None,None,None,None,None
4,152820f0-6ca9-4df3-943b-a67d568ff472,PASS,1,0 days 00:00:00.340000,0 days 00:00:02.869454,alive,768,768,99174,0.499564,...,0.48318,3468,KICK_OFF,RIGHT_FOOT,None,None,None,None,None,None


In [8]:
# Display dataset info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3415 entries, 0 to 3414
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   event_id            3415 non-null   object         
 1   event_type          3415 non-null   object         
 2   period_id           3415 non-null   int64          
 3   timestamp           3415 non-null   timedelta64[ns]
 4   end_timestamp       1676 non-null   timedelta64[ns]
 5   ball_state          3415 non-null   object         
 6   ball_owning_team    3415 non-null   object         
 7   team_id             3415 non-null   object         
 8   player_id           3400 non-null   object         
 9   coordinates_x       3389 non-null   float64        
 10  coordinates_y       3389 non-null   float64        
 11  result              1844 non-null   object         
 12  success             1844 non-null   object         
 13  end_coordinates_x   1701 non-null

## Filter columns

In [9]:
# Filter columns from the dataset
filtered_df = dataset.to_df(
    "player_id",
    "player",  # type: ignore
    "team_id",
    "team",
    "event_id",
    "event_type",
    "result",
    "success",
    "body_part_type",
    "pass_type",
    "duel_type",
    "set_piece_type",
    "goalkeeper_type",
    "card_type",
    "coordinates_x",
    "coordinates_y",
    "time",
)

In [10]:
# Create a mapping for dtypes of all columns
dtype_mapping = {
    "player_id": "Int64",
    "player": "string",
    "team_id": "Int64",
    "team": "string",
    "event_id": "string",
    "event_type": "category",
    "result": "category",
    "success": "boolean",
    "body_part_type": "category",
    "pass_type": "category",
    "duel_type": "category",
    "set_piece_type": "category",
    "goalkeeper_type": "category",
    "card_type": "category",
    "coordinates_x": "Float64",
    "coordinates_y": "Float64",
    "time": "string",
}

# Convert the type of all DataFrame columns
filtered_df = filtered_df.astype(dtype_mapping)

# Verify changes
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3415 entries, 0 to 3414
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   player_id        3400 non-null   Int64   
 1   player           3400 non-null   string  
 2   team_id          3415 non-null   Int64   
 3   team             3415 non-null   string  
 4   event_id         3415 non-null   string  
 5   event_type       3415 non-null   category
 6   result           1844 non-null   category
 7   success          1844 non-null   boolean 
 8   body_part_type   938 non-null    category
 9   pass_type        227 non-null    category
 10  duel_type        109 non-null    category
 11  set_piece_type   82 non-null     category
 12  goalkeeper_type  11 non-null     category
 13  card_type        8 non-null      category
 14  coordinates_x    3389 non-null   Float64 
 15  coordinates_y    3389 non-null   Float64 
 16  time             3415 non-null   string  


In [11]:
# Display DataFrame
filtered_df.head()

,player_id,player,team_id,team,event_id,event_type,result,success,body_part_type,pass_type,duel_type,set_piece_type,goalkeeper_type,card_type,coordinates_x,coordinates_y,time
0,<NA>,<NA>,772,Spain,50aa204f-5d65-4145-8597-5d5628fb7898,GENERIC:Starting XI,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
1,<NA>,<NA>,768,England,a279cbee-9ab3-4cfb-9c51-27cacc1bf2a2,GENERIC:Starting XI,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
2,<NA>,<NA>,768,England,d2126e70-9f04-4bb7-ba2b-9377836d1757,GENERIC:Half Start,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
3,<NA>,<NA>,772,Spain,54d78bfa-4146-42bd-acdc-97bcd393dd81,GENERIC:Half Start,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
4,99174,Kobbie Mainoo,768,England,152820f0-6ca9-4df3-943b-a67d568ff472,PASS,COMPLETE,True,RIGHT_FOOT,NaN,NaN,KICK_OFF,NaN,NaN,0.499564,0.499327,P1T00:00


In [12]:
# Filter DataFrame to only keeps rows related to player events
players_df = filtered_df[filtered_df["player_id"].notna()]
players_df.head()

,player_id,player,team_id,team,event_id,event_type,result,success,body_part_type,pass_type,duel_type,set_piece_type,goalkeeper_type,card_type,coordinates_x,coordinates_y,time
4,99174,Kobbie Mainoo,768,England,152820f0-6ca9-4df3-943b-a67d568ff472,PASS,COMPLETE,True,RIGHT_FOOT,NaN,NaN,KICK_OFF,NaN,NaN,0.499564,0.499327,P1T00:00
5,3468,Jordan Pickford,768,England,d64668c7-747c-4a7d-912c-e1c3ff357a67,GENERIC:Ball Receipt*,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,0.21834,0.48318,P1T00:03
6,3468,Jordan Pickford,768,England,9c107df3-a3c8-4ad5-bc35-00214087a105,CARRY,COMPLETE,True,NaN,NaN,NaN,NaN,NaN,NaN,0.21834,0.48318,P1T00:03
7,3468,Jordan Pickford,768,England,237201b8-aef8-4823-b282-e82875795c07,PASS,OUT,False,LEFT_FOOT,LONG_BALL,NaN,NaN,NaN,NaN,0.244381,0.386189,P1T00:05
8,22084,Bukayo Saka,768,England,c979e198-edc1-4f22-851a-26cedb6474cf,GENERIC:Ball Receipt*,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,0.798231,0.721654,P1T00:10


In [13]:
# Get players_df info
players_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3400 entries, 4 to 3412
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   player_id        3400 non-null   Int64   
 1   player           3400 non-null   string  
 2   team_id          3400 non-null   Int64   
 3   team             3400 non-null   string  
 4   event_id         3400 non-null   string  
 5   event_type       3400 non-null   category
 6   result           1844 non-null   category
 7   success          1844 non-null   boolean 
 8   body_part_type   938 non-null    category
 9   pass_type        227 non-null    category
 10  duel_type        109 non-null    category
 11  set_piece_type   82 non-null     category
 12  goalkeeper_type  11 non-null     category
 13  card_type        8 non-null      category
 14  coordinates_x    3387 non-null   Float64 
 15  coordinates_y    3387 non-null   Float64 
 16  time             3400 non-null   string  
dtype

## Obtain minutes played for every player

In [14]:
# Aggregate dataset to obtain minutes played for every player
minutes_dataset = dataset.aggregate("minutes_played")

In [15]:
# Display minutes played for every player
for entry in minutes_dataset:
    print(
        f"{entry.player}:\n"
        f"    -> {entry.start_time} - {entry.end_time}\n"
        f"    -> {entry.duration.total_seconds() / 60:02.0f}:{entry.duration.total_seconds() % 60:02.0f}"
    )

Mikel Merino Zazón:
    -> P2T43:41 - P2T49:02
    -> 05:21
Álvaro Borja Morata Martín:
    -> P1T00:00 - P2T22:19
    -> 69:22
Aymeric Laporte:
    -> P1T00:00 - P2T49:02
    -> 96:04
José Ignacio Fernández Iglesias:
    -> P2T37:38 - P2T49:02
    -> 11:24
Daniel Carvajal Ramos:
    -> P1T00:00 - P2T49:02
    -> 96:04
Fabián Ruiz Peña:
    -> P1T00:00 - P2T49:02
    -> 96:04
Mikel Oyarzabal Ugarte:
    -> P2T22:19 - P2T49:02
    -> 27:43
Rodrigo Hernández Cascante:
    -> P1T00:00 - P2T00:00
    -> 47:03
Unai Simón Mendibil:
    -> P1T00:00 - P2T49:02
    -> 96:04
Daniel Olmo Carvajal:
    -> P1T00:00 - P2T49:02
    -> 96:04
Marc Cucurella Saseta:
    -> P1T00:00 - P2T49:02
    -> 96:04
Robin Aime Robert Le Normand:
    -> P1T00:00 - P2T37:38
    -> 85:41
Martín Zubimendi Ibáñez:
    -> P2T00:00 - P2T49:02
    -> 49:02
Nicholas Williams Arthuer:
    -> P1T00:00 - P2T49:02
    -> 96:04
Lamine Yamal Nasraoui Ebana:
    -> P1T00:00 - P2T43:41
    -> 91:43
Kyle Walker:
    -> P1T00:00 - P

## Extract player and team metrics

### Helper functions

In [16]:
def get_player_and_opponent_teams(dataset: EventDataset, player_df: pd.DataFrame) -> tuple[str, str]:
    """Get player's team and opponent team from player's DataFrame"""
    # Obtain both team names from the dataset
    home_team, away_team = dataset.metadata.teams
    home_team_name, away_team_name = home_team.name, away_team.name

    # Get player's team
    player_team = player_df["team"].iloc[0]

    # Get opponent team
    if player_team == home_team_name:
        opponent_team = away_team_name
    else:
        opponent_team = home_team_name

    return player_team, opponent_team

In [17]:
def get_player_position(dataset: EventDataset, player_name: str) -> str:
    """Determine main position for a player based on time spent in each position"""

    # Set default position and duration
    player_position = ("", 0)

    # Iterate through dataset to find player's position history
    for team in dataset.metadata.teams:
        for player in team.players:
            # Only get position for the specified player
            if player.name == player_name:
                for start_time, end_time, position in player.positions.ranges():
                    duration = (end_time - start_time).total_seconds()

                    # Navigate to obtain the position group
                    while position.parent is not None:
                        position = position.parent

                    # Compare and update the variable if this position has longer duration
                    prev_position, prev_duration = player_position
                    if position.name != prev_position and duration > prev_duration:
                        player_position = (position.name, duration)

    # Return only the position name
    return player_position[0]

In [18]:
def get_player_minutes(minutes_dataset: list, player_name: str) -> int:
    """Get minutes played by a player"""

    # Initialize minutes played
    minutes_played = 0

    # Iterate through minutes dataset to find the player
    for entry in minutes_dataset:
        if entry.player.name == player_name:
            minutes_played = round(entry.duration.total_seconds() / 60)
            break

    return minutes_played

In [19]:
def get_team_minutes(minutes_dataset: list, team_name: str) -> int:
    """Get total minutes played by a team"""

    # Initialize total minutes
    total_minutes = 0

    # Iterate through minutes dataset to find players in the specified team
    for entry in minutes_dataset:
        if entry.player.team.name == team_name:
            total_minutes += round(entry.duration.total_seconds() / 60)

    return total_minutes

In [20]:
def calculate_assists_for_player(player_data: pd.DataFrame, df: pd.DataFrame) -> int:
    """Calculate assists by looking for SHOT_ASSIST events before goals"""

    # Get all goal events in the match
    goals_df = df[df["event_type"] == "SHOT"][df["result"] == "GOAL"]

    # Get player shot assist events
    shot_assists_df = player_data[player_data["pass_type"] == "SHOT_ASSIST"]

    # Get indexes of DataFrames
    event_indexes = df.index.to_list()
    goal_indexes = goals_df.index.to_list()
    shot_assist_indexes = shot_assists_df.index.to_list()

    # Initialize assist counter
    assists_count = 0

    # Iterate through each shot assist and look for a goal in subsequent events
    for shot_assist_idx in shot_assist_indexes:
        event_idx = shot_assist_idx

        while event_idx in event_indexes:
            event_idx += 1
            event = df.iloc[event_idx]

            if event_idx in goal_indexes:  # Found a goal for this shot assist
                assists_count += 1
                break
            elif event["event_type"] == "SHOT":  # Found an unsuccessful shot
                break

    return assists_count

In [21]:
def calculate_tackle_win_ratio(team_df: pd.DataFrame) -> float:
    """Calculate tackle win ratio for a team"""

    total_duels = len(team_df[team_df["event_type"] == "DUEL"])
    successful_duels = len(team_df[team_df["event_type"] == "DUEL"][team_df["success"]])

    if total_duels == 0:
        return 0.0
    else:
        return round(successful_duels / total_duels, 2)

### Player metrics

In [22]:
def extract_player_metrics(dataset: EventDataset, minutes_dataset: list, df: pd.DataFrame):
    """Extract player metrics: team, position, minutes played, goals, assists, crosses, dribbles and passes"""

    # Initialize dictionary to hold player metrics
    player_metrics = {}

    for player_name in df["player"].unique():
        # Filter events data for the specific player
        player_df = df[df["player"] == player_name]

        # Obtain individual metrics
        player_team, opponent_team = get_player_and_opponent_teams(dataset, player_df)
        player_position = get_player_position(dataset, player_name)
        player_minutes = get_player_minutes(minutes_dataset, player_name)
        player_goals = len(player_df[player_df["event_type"] == "SHOT"][player_df["result"] == "GOAL"])
        player_assists = calculate_assists_for_player(player_df, df)
        player_crosses = len(player_df[player_df["pass_type"] == "CROSS"][player_df["success"]])
        player_dribbles = len(player_df[player_df["event_type"] == "TAKE_ON"][player_df["success"]])
        player_passes = len(player_df[player_df["event_type"] == "PASS"][player_df["success"]])

        # Store metrics in dictionary
        player_metrics[player_name] = {
            "team": player_team,
            "opponent_team": opponent_team,
            "position": player_position,
            "minutes_played": player_minutes,
            "goals": player_goals,
            "assists": player_assists,
            "crosses": player_crosses,
            "dribbles": player_dribbles,
            "passes": player_passes,
        }

    return player_metrics

In [23]:
# Extract and display player metrics
player_metrics = extract_player_metrics(dataset, minutes_dataset, players_df)
print(f"Player metrics extracted for {len(player_metrics)} players\n")

print("Player metrics:")
for k, v in player_metrics.items():
    print(f" -> {k}")
    print(f"    {v}")

Player metrics extracted for 29 players

Player metrics:
 -> Kobbie Mainoo
    {'team': 'England', 'opponent_team': 'Spain', 'position': 'Midfielder', 'minutes_played': 72, 'goals': 0, 'assists': 0, 'crosses': 0, 'dribbles': 0, 'passes': 14}
 -> Jordan Pickford
    {'team': 'England', 'opponent_team': 'Spain', 'position': 'Goalkeeper', 'minutes_played': 96, 'goals': 0, 'assists': 0, 'crosses': 0, 'dribbles': 0, 'passes': 26}
 -> Bukayo Saka
    {'team': 'England', 'opponent_team': 'Spain', 'position': 'Midfielder', 'minutes_played': 96, 'goals': 0, 'assists': 0, 'crosses': 1, 'dribbles': 0, 'passes': 21}
 -> Unai Simón Mendibil
    {'team': 'Spain', 'opponent_team': 'England', 'position': 'Goalkeeper', 'minutes_played': 96, 'goals': 0, 'assists': 0, 'crosses': 0, 'dribbles': 0, 'passes': 34}
 -> Robin Aime Robert Le Normand
    {'team': 'Spain', 'opponent_team': 'England', 'position': 'Defender', 'minutes_played': 85, 'goals': 0, 'assists': 0, 'crosses': 0, 'dribbles': 0, 'passes': 80}

### Team metrics

In [24]:
spain_df = players_df[players_df["team"] == "Spain"]
england_df = players_df[players_df["team"] == "England"]

In [25]:
def extract_team_metrics(minutes_played_dataset: list, df: pd.DataFrame):
    """Extract team metrics: minutes played, goals, yellow and red cards, interceptions, clearances and tackle wins"""

    # Initialize dictionary to hold team metrics
    team_metrics = {}

    for team in df["team"].unique():
        team_df = df[df["team"] == team]

        # Obtain team metrics
        team_minutes = get_team_minutes(minutes_played_dataset, team)
        team_goals = len(team_df[team_df["event_type"] == "SHOT"][team_df["result"] == "GOAL"])
        team_yellow_cards = len(team_df[team_df["event_type"] == "CARD"][team_df["card_type"] == "FIRST_YELLOW"])
        team_red_cards = len(
            team_df[team_df["event_type"] == "CARD"][team_df["card_type"].isin(["RED", "SECOND_YELLOW"])]
        )
        team_interceptions = len(team_df[team_df["event_type"] == "INTERCEPTION"][team_df["success"]])
        team_clearances = len(team_df[team_df["event_type"] == "CLEARANCE"])
        team_tackle_win_ratio = calculate_tackle_win_ratio(team_df)

        # Store metrics in dictionary
        team_metrics[team] = {
            "total_minutes": team_minutes,
            "goals": team_goals,
            "yellow_cards": team_yellow_cards,
            "red_cards": team_red_cards,
            "interceptions": team_interceptions,
            "clearances": team_clearances,
            "tackle_win_ratio": team_tackle_win_ratio,
        }

    return team_metrics

In [26]:
# Extract and display team metrics
team_metrics = extract_team_metrics(minutes_dataset, players_df)
print(f"Team metrics extracted for {len(team_metrics)} teams\n")

print("Team metrics:")
for k, v in team_metrics.items():
    print(f" -> {k}")
    print(f"    {v}")

Team metrics extracted for 2 teams

Team metrics:
 -> England
    {'total_minutes': 1056, 'goals': 1, 'yellow_cards': 3, 'red_cards': 0, 'interceptions': 6, 'clearances': 25, 'tackle_win_ratio': 0.51}
 -> Spain
    {'total_minutes': 1056, 'goals': 2, 'yellow_cards': 1, 'red_cards': 0, 'interceptions': 4, 'clearances': 18, 'tackle_win_ratio': 0.54}


## Calculate the Player Performace Index

In [27]:
def calculate_ppi_for_players(player_metrics: dict, team_metrics: dict) -> pd.DataFrame:
    """Calculate EA Sports PPI for each player"""

    # Initialize list to hold player PPI scores
    ppi_list = []

    for player, metrics in player_metrics.items():
        # Get metrics needed for calculations
        player_team = metrics["team"]
        player_position = metrics["position"]
        player_goals = metrics["goals"]
        player_assists = metrics["assists"]
        player_crosses = metrics["crosses"]
        player_dribbles = metrics["dribbles"]
        player_passes = metrics["passes"]

        opponent_team = metrics["opponent_team"]
        opponent_metrics = team_metrics[opponent_team]
        opp_yellows = opponent_metrics["yellow_cards"]
        opp_reds = opponent_metrics["red_cards"]
        opp_interceptions = opponent_metrics["interceptions"]
        opp_clearances = opponent_metrics["clearances"]
        opp_tackle_win_ratio = opponent_metrics["tackle_win_ratio"]

        team_goals = team_metrics[player_team]["goals"]
        opponent_goals = team_metrics[opponent_team]["goals"]

        player_minutes = metrics["minutes_played"]
        team_minutes = team_metrics[player_team]["total_minutes"]
        minutes_ratio = round(player_minutes / team_minutes, 2)

        # Subindex 1: Modelling Match Outcome
        MODEL_COEFFICIENTS = {
            "crosses": 0.519,
            "dribbles": 0.118,
            "passes": 0.034,
            "opp_interceptions": -0.024,
            "opp_yellows": 0.253,
            "opp_reds": 1.023,
            "opp_tackle_win_ratio": -0.170,
            "opp_clearances": -0.017,
            "constant": 6.463,
        }

        index_1 = MODEL_COEFFICIENTS["constant"]
        index_1 += player_crosses * MODEL_COEFFICIENTS["crosses"]
        index_1 += player_dribbles * MODEL_COEFFICIENTS["dribbles"]
        index_1 += player_passes * MODEL_COEFFICIENTS["passes"]
        index_1 += opp_interceptions * MODEL_COEFFICIENTS["opp_interceptions"]
        index_1 += opp_yellows * MODEL_COEFFICIENTS["opp_yellows"]
        index_1 += opp_reds * MODEL_COEFFICIENTS["opp_reds"]
        index_1 += opp_tackle_win_ratio * MODEL_COEFFICIENTS["opp_tackle_win_ratio"]
        index_1 += opp_clearances * MODEL_COEFFICIENTS["opp_clearances"]

        # Subindex 2: Points-Sharing Index
        POINTS_FOR_WIN = 3
        POINTS_FOR_DRAW = 1
        POINTS_FOR_LOSS = 0

        if team_goals > opponent_goals:
            index_2 = minutes_ratio * POINTS_FOR_WIN
        elif team_goals == opponent_goals:
            index_2 = minutes_ratio * POINTS_FOR_DRAW
        else:
            index_2 = minutes_ratio * POINTS_FOR_LOSS

        # Subindex 3: Appearance Index
        POINTS_PER_GAME = 1.34
        index_3 = minutes_ratio * POINTS_PER_GAME

        # Subindex 4: Goal-Scoring Index
        POINTS_PER_GOAL = 1.039
        index_4 = player_goals * POINTS_PER_GOAL

        # Subindex 5: Assists Index
        POINTS_PER_ASSIST = 1.039
        index_5 = player_assists * POINTS_PER_ASSIST

        # Subindex 6: Clean-Sheets Index
        POINTS_PER_CLEAN_SHEET_GOALKEEPER = 0.585
        POINTS_PER_CLEAN_SHEET_DEFENDER = 0.364
        POINTS_PER_CLEAN_SHEET_MIDFIELDER = 0.150
        POINTS_PER_CLEAN_SHEET_STRIKER = 0.071

        if opponent_goals == 0:
            match player_position:
                case "Goalkeeper":
                    index_6 = POINTS_PER_CLEAN_SHEET_GOALKEEPER
                case "Defender":
                    index_6 = POINTS_PER_CLEAN_SHEET_DEFENDER
                case "Midfielder":
                    index_6 = POINTS_PER_CLEAN_SHEET_MIDFIELDER
                case "Attacker":
                    index_6 = POINTS_PER_CLEAN_SHEET_STRIKER
                case _:
                    index_6 = 0
        else:
            index_6 = 0

        # Final PPI calculation
        I1_WEIGHT = 0.25
        I2_WEIGHT = 0.375
        I3_WEIGHT = 0.125
        I4_WEIGHT = 0.125
        I5_WEIGHT = 0.0625
        I6_WEIGHT = 0.0625

        player_index = round(
            100
            * (
                I1_WEIGHT * index_1
                + I2_WEIGHT * index_2
                + I3_WEIGHT * index_3
                + I4_WEIGHT * index_4
                + I5_WEIGHT * index_5
                + I6_WEIGHT * index_6
            )
        )

        # Append player PPI data to list
        ppi_list.append(
            {
                "player": player,
                "team": player_team,
                "position": player_position,
                "index_score": player_index,
                "minutes_played": player_minutes,
                "goals": player_goals,
                "assists": player_assists,
                "crosses": player_crosses,
                "dribbles": player_dribbles,
                "passes": player_passes,
            }
        )

    # Convert list to DataFrame
    ppi_df = pd.DataFrame(ppi_list)
    return ppi_df

In [28]:
# Calculate PPI scores
ppi_df = calculate_ppi_for_players(player_metrics, team_metrics)

# Convert to DataFrame and rank players
ppi_df = ppi_df.sort_values("index_score", ascending=False).reset_index(drop=True)
ppi_df.insert(0, "rank", range(1, len(ppi_df) + 1))

# Display all players ranked by PPI
ppi_df

,rank,player,team,position,index_score,minutes_played,goals,assists,crosses,dribbles,passes
0,1,Aymeric Laporte,Spain,Defender,244,96,0,0,0,0,80
1,2,Robin Aime Robert Le Normand,Spain,Defender,242,85,0,0,0,0,80
2,3,Daniel Carvajal Ramos,Spain,Defender,232,96,0,0,0,1,63
3,4,Nicholas Williams Arthuer,Spain,Midfielder,227,96,1,0,0,1,41
4,5,Fabián Ruiz Peña,Spain,Midfielder,226,96,0,0,0,1,56
5,6,Marc Cucurella Saseta,Spain,Defender,218,96,0,1,0,0,42
6,7,Unai Simón Mendibil,Spain,Goalkeeper,205,96,0,0,0,0,34
7,8,Lamine Yamal Nasraoui Ebana,Spain,Midfielder,203,91,0,1,0,0,24
8,9,Daniel Olmo Carvajal,Spain,Midfielder,197,96,0,0,0,0,25
9,10,Rodrigo Hernández Cascante,Spain,Midfielder,194,47,0,0,0,0,29
